# T20 Oracle - model training

Features come from `backend/app/feature_eng.py` (Cricsheet JSON files in `../data/`).
Matches are processed in date order, every feature is computed from pre-match state only,
no-result matches are skipped, and evaluation is on a chronological holdout (the most recent
20% of matches), with the encoder and scaler fit on the training set only. See the README's
Results and limitations section for why (the original pipeline leaked the match being
predicted into its own form features).

The hyperparameter grids below are a trimmed version of the original ones (same families and
ranges) so the notebook runs in a couple of minutes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import joblib
import os
import sys

In [ ]:
# Build the feature CSV from the Cricsheet JSONs in ../data/ if it is not there yet.
# feature_eng.py processes matches in date order and uses pre-match state only.
CSV_PATH = "../data/t20_features_full.csv"
if not os.path.exists(CSV_PATH):
    sys.path.append("../backend/app")
    from feature_eng import load_all_matches, feature_engineering
    matches = load_all_matches("../data")
    feature_engineering(matches).to_csv(CSV_PATH, index=False)

df = pd.read_csv(CSV_PATH)
df.drop(['winner', 'date'], inplace=True, axis=1)
print(f"{len(df)} matches (no-result games are skipped by feature_eng.py)")

In [ ]:
# Chronological split: train on the older 80% of matches, test on the most recent 20%.
# A random split lets the model train on the future and test on the past.
split = int(len(df) * 0.8)
train_df, test_df = df.iloc[:split], df.iloc[split:]

cat_cols = ['teamA', 'teamB', 'venue', 'toss_winner', 'toss_decision', 'competition']
num_cols = [c for c in df.columns if c not in cat_cols + ['match_winner_teamA']]

# Fit preprocessing on TRAINING data only; unseen categories at inference map to -1.
label_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_cat = label_encoder.fit_transform(train_df[cat_cols])
X_test_cat = label_encoder.transform(test_df[cat_cols])

scaler = StandardScaler().fit(train_df[num_cols])
X_train = np.hstack([X_train_cat, scaler.transform(train_df[num_cols])])
X_test = np.hstack([X_test_cat, scaler.transform(test_df[num_cols])])
y_train = train_df['match_winner_teamA'].values
y_test = test_df['match_winner_teamA'].values

MODELS_DIR = "../backend/models/"
os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(label_encoder, os.path.join(MODELS_DIR, 'label_encoders.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.pkl'))
print(f"Train: {len(y_train)} matches, test: {len(y_test)} (most recent). Preprocessing fit on train only.")

In [ ]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("LogReg test accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
rf_params = {
    'n_estimators': [100, 300],
    'max_depth': [4, 8, None],
    'min_samples_split': [2, 10],
    'max_features': ['sqrt']
}

rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, rf_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

print("Best Random Forest CV accuracy:", rf_grid.best_score_)
print("Best RF Params:", rf_grid.best_params_)

In [ ]:
xgb_params = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_grid = GridSearchCV(xgb, xgb_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
xgb_grid.fit(X_train, y_train)

print("Best XGBoost CV accuracy:", xgb_grid.best_score_)
print("Best XGB Params:", xgb_grid.best_params_)

In [ ]:
best_rf = rf_grid.best_estimator_
best_xgb = xgb_grid.best_estimator_

for name, m in [("LogReg", model), ("RF", best_rf), ("XGB", best_xgb)]:
    proba = m.predict_proba(X_test)
    print(f"{name}: accuracy={accuracy_score(y_test, (proba[:,1] > 0.5)):.4f}, "
          f"log_loss={log_loss(y_test, proba):.4f}, "
          f"brier={brier_score_loss(y_test, proba[:, 1]):.4f}")

In [ ]:
prod_model = xgb_grid.best_estimator_
joblib.dump(prod_model, '../backend/models/matchWinner.pkl')
print("Model saved to ../backend/models/matchWinner.pkl")